# Face Attribute Detection with UniFace

<div style="display:flex; flex-wrap:wrap; align-items:center;">
  <a style="margin-right:10px; margin-bottom:6px;" href="https://pepy.tech/projects/uniface"><img alt="PyPI Downloads" src="https://static.pepy.tech/personalized-badge/uniface?period=total&units=international_system&left_color=grey&right_color=blue&left_text=Downloads"></a>
  <a style="margin-right:10px; margin-bottom:6px;" href="https://pypi.org/project/uniface/"><img alt="PyPI Version" src="https://img.shields.io/pypi/v/uniface.svg"></a>
  <a style="margin-right:10px; margin-bottom:6px;" href="https://opensource.org/licenses/MIT"><img alt="License" src="https://img.shields.io/badge/License-MIT-blue.svg"></a>
  <a style="margin-bottom:6px;" href="https://github.com/yakhyo/uniface"><img alt="GitHub Stars" src="https://img.shields.io/github/stars/yakhyo/uniface.svg?style=social"></a>
</div>

**UniFace** is a lightweight, production-ready Python library for face detection, recognition, tracking, landmark analysis, face parsing, gaze estimation, and face attributes.

🔗 **GitHub**: [github.com/yakhyo/uniface](https://github.com/yakhyo/uniface) | 📚 **Docs**: [yakhyo.github.io/uniface](https://yakhyo.github.io/uniface)

---

This notebook demonstrates face attribute detection with **FaceAttribNet** — five independent binary face states per face: left/right eye openness, eyeglasses, face mask, and sunglasses.

## 1. Install UniFace

In [ ]:
%pip install -q "uniface[cpu]"

# Clone repo for assets (Colab only)
import os
if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
    if not os.path.exists('uniface'):
        !git clone --depth 1 https://github.com/yakhyo/uniface.git
    os.chdir('uniface/examples')

## 2. Import Libraries

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

import uniface
from uniface.attribute import FaceAttribNet
from uniface.detection import RetinaFace

print(f"UniFace version: {uniface.__version__}")

## 3. Initialize Models

In [ ]:
# Initialize face detector
detector = RetinaFace(confidence_threshold=0.5)

# Initialize face attribute model
face_attrib = FaceAttribNet()

print("Models initialized successfully!")

## 4. Process All Test Images

Each face gets a `FaceStateResult` with five **independent** probabilities: `left_eye_open`, `right_eye_open`, `eyeglasses`, `mask`, `sunglasses`.

> ⚠️ These come from independent binary heads — they do not sum to 1 and several can be high at once (a face can wear both sunglasses and a mask). Threshold each attribute separately; never `argmax`.

In [ ]:
THRESHOLD = 0.5


def annotate(image, face, result):
    """Draw the face box and the attributes that exceed THRESHOLD."""
    x1, y1, x2, y2 = map(int, face.bbox)
    cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)

    scale = max(0.6, (x2 - x1) / 250)
    for i, label in enumerate(result.labels(THRESHOLD)):
        y_text = y1 + int((i + 1) * 32 * scale)
        cv2.putText(image, label, (x2 + 8, y_text), cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 255, 0), 2, cv2.LINE_AA)


test_images_dir = Path('../assets/test_images/attributes')
test_images = sorted(test_images_dir.glob('*.jpg'))

original_images = []
annotated_images = []
titles = []

for img_path in test_images:
    image = cv2.imread(str(img_path))
    if image is None:
        continue

    original_images.append(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

    faces = detector.detect(image)
    for face in faces:
        result = face_attrib.predict(image, face)
        active = ', '.join(result.labels(THRESHOLD)) or 'none'
        print(f"{img_path.name}: {active}")
        annotate(image, face, result)

    annotated_images.append(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    titles.append(img_path.stem)

print(f"\nProcessed {len(original_images)} images")

## 5. Visualize Results

**First row**: Original images
**Second row**: Faces annotated with the attributes above the threshold

In [ ]:
num_images = len(original_images)

fig, axes = plt.subplots(2, num_images, figsize=(4 * num_images, 9))

if num_images == 1:
    axes = axes.reshape(2, 1)

for i in range(num_images):
    axes[0, i].imshow(original_images[i])
    axes[0, i].set_title(titles[i], fontsize=12)
    axes[0, i].axis('off')

    axes[1, i].imshow(annotated_images[i])
    axes[1, i].set_title('Attributes', fontsize=12)
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

## 6. Inspect a Single Prediction

`predict()` returns a `FaceStateResult` and also writes the probabilities back onto the `Face` object (`face.left_eye_open`, `face.right_eye_open`, `face.eyeglasses`, `face.mask`, `face.sunglasses`).

In [ ]:
image = cv2.imread(str(test_images_dir / 'sunglasses.jpg'))
face = detector.detect(image)[0]

result = face_attrib.predict(image, face)

print(result)
print()
for name, prob in result.as_dict().items():
    marker = '✔' if prob > THRESHOLD else '✘'
    print(f"  {marker} {name:15s} {prob:.4f}")

print()
print(f"Face enriched: sunglasses={face.sunglasses:.4f}, left_eye_open={face.left_eye_open:.4f}")

## Notes

- **Input**: full image + detected `Face` (the model crops by `face.bbox` internally, letterboxes to 128×128)
- **Output**: `FaceStateResult` with five independent probabilities in `[0, 1]`
- **Multi-label**: threshold each attribute separately (`result.labels(threshold)`); never `argmax`
- **Face enrichment**: `predict()` also sets `face.left_eye_open`, `face.right_eye_open`, `face.eyeglasses`, `face.mask`, `face.sunglasses`
- **Model**: Qualcomm's [FaceAttribNet](https://github.com/qualcomm/ai-hub-models/tree/main/src/qai_hub_models/models/face_attrib_net) ("Facial-Attribute-Detection"), ONNX export from [yakhyo/face-attribute](https://github.com/yakhyo/face-attribute)

### With FaceAnalyzer

```python
from uniface import FaceAnalyzer, FaceAttribNet

analyzer = FaceAnalyzer(predictors=[FaceAttribNet()])
faces = analyzer.analyze(image)

for face in faces:
    print(face.eyeglasses, face.mask, face.sunglasses)
```